# Fase 2 · Transformación de datos en los datasets

## Objetivo

El objetivo de esta fase consiste en ejecutar los cambios y ajustes detectados durante el Análisis Exploratorio de Datos (EDA), para unificar, limpiar y transformar los datasets seleccionados en este proyecto.

En esta etapa se trabaja principalmente en:

- abordar los duplicados,
- combinar varios datasets,
- gestionar los valores nulos,
- homogenizar categorías para asegurar la integridad semántica,
- y exportar los archivos finales ya depurados.

Este proceso permitirá tener un conjunto de datos más consistentes, comparables y listos para la siguiente fase de análisis avanzado y visualización.

In [60]:
# Importación de librerías
import pandas as pd
import numpy as np

# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de transformación 
from src.etl.load_data import load_friends_data_raw
from src.etl import transform as trans

# Configuración para visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None) 

In [61]:
dfs = load_friends_data_raw()

[2026-05-21 17:06:27] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_raw
[2026-05-21 17:06:27] INFO - → Cargando bodas_divorcios_ross.csv...
[2026-05-21 17:06:27] INFO - → Cargando cameos_friends_completo.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends_emotions.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends_episodes_v2.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends_episodes_v3.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends_escenarios.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends_info.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends_quotes.csv...
[2026-05-21 17:06:27] INFO - → Cargando friends.csv...
[2026-05-21 17:06:27] INFO - → Cargando phoebe_buffay_songs.csv...
[2026-05-21 17:06:27] INFO - → Cargando apariciones_detalladas_pato_y_pollito.csv...
[2026-05-21 17:06:27] INFO - Todos los datasets fueron cargados correctamente.


## 1. Transformación de  las variables numéricas (friends_quotes) de orden de float a entero (int) para mejorar la estructura. 

In [62]:
df_quotes = dfs["quotes"]

df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0.0,1.0
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1.0,1.0
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2.0,1.0
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3.0,1.0
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4.0,1.0


In [63]:
df_quotes["quote_order"] = df_quotes["quote_order"].astype(int)
df_quotes["season"] = df_quotes["season"].astype(int)


In [64]:
df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0,1
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1,1
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2,1
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3,1
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4,1


## 2. Limpiar y estandarizar la columna written_by (friends_info)

In [65]:
df_info= dfs["info"]

df_info.head()

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,David Crane & Marta Kauffman,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,David Crane & Marta Kauffman,1994-09-29,20.2,8.1
2,1,3,The One with the Thumb,James Burrows,Jeffrey Astrof & Mike Sikowitz,1994-10-06,19.5,8.2
3,1,4,The One with George Stephanopoulos,James Burrows,Alexa Junge,1994-10-13,19.7,8.1
4,1,5,The One with the East German Laundry Detergent,Pamela Fryman,Jeff Greenstein & Jeff Strauss,1994-10-20,18.6,8.5


In [66]:
#Limpiamos el caracter de espacio en blanco antes de dividir.
df_info["written_by"] = df_info["written_by"].str.replace("\u200a", "", regex=False)


In [67]:
#Dividimos en story_by y teleplay_by la columna "written_by" usando expresiones regulares para extraer los nombres de los escritores.

df_info["story_by"] = df_info["written_by"].str.extract(r"Story by:\s*(.*?)\s*(?=Teleplay by:|$)")
df_info["teleplay_by"] = df_info["written_by"].str.extract(r"Teleplay by:\s*(.*)")


#Eliminamos los espacios en blanco alrededor de los nombres de los escritores y separamos en dos columnas si hay más de un escritor
df_info[["written_by1", "written_by2"]] = (
    df_info["written_by"]
    .fillna("&")                     # evita errores por NaN
    .str.split("&", n=1, expand=True)  # divide SOLO en 2 partes
    .apply(lambda col: col.str.strip()) # limpia espacios
)



# 1. Limpiar unicode raro
df_info["written_by"] = df_info["written_by"].str.replace("\u200a", "", regex=False)

# 2. Extraer Story by
df_info["story_by"] = df_info["written_by"].str.extract(
    r"Story by:\s*(.*?)\s*(?=Teleplay by:|$)"
)

# 3. Extraer Teleplay by
df_info["teleplay_by"] = df_info["written_by"].str.extract(
    r"Teleplay by:\s*(.*)"
)

# 4. Limpiar espacios
df_info["story_by"] = df_info["story_by"].str.strip()
df_info["teleplay_by"] = df_info["teleplay_by"].str.strip()

# 5. Para celdas sin Story/Teleplay → meter todo en story_by
mask_no_tags = df_info["written_by"].str.contains("Story by|Teleplay by", case=False, na=False) == False
df_info.loc[mask_no_tags, "story_by"] = df_info.loc[mask_no_tags, "written_by"]
df_info.loc[mask_no_tags, "teleplay_by"] = None


In [69]:
df_info.loc[112, "written_by"]


'Story by: Alicia Sky VarinaitisTeleplay by: Gigi McCreery & Perry Rein'

In [70]:
df_info.loc[117, "written_by"]

'Story by: Scott SilveriTeleplay by: Gregory S. Malins'

In [71]:
df_info.loc[212, "written_by"]

'Story by: Dana Klein BorkowTeleplay by: Mark Kunerth'

In [72]:
#Creo que story quién inventó la historia y luego "Teleplay" --> quién escribió el guión final.

In [74]:
df_info.sample(10)

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating,story_by,teleplay_by
162,7,17,The One with the Cheap Wedding Dress,Kevin S. Bright,Story by: Brian Buckner & Sebastian JonesTelep...,2001-03-15,20.84,8.4,Brian Buckner & Sebastian Jones,Andrew Reich & Ted Cohen
182,8,13,The One Where Chandler Takes a Bath,Ben Weiss,Vanessa McCarthy,2002-01-17,29.24,8.6,NaN,NaN
204,9,11,The One Where Rachel Goes Back to Work,Gary Halvorson,Story by: Judd RubinTeleplay by: Peter Tibbals,2003-01-09,23.67,8.0,Judd Rubin,Peter Tibbals
38,2,15,The One Where Ross and Rachel...You Know,Michael Lembeck,Michael Curtis & Gregory S. Malins,1996-02-08,32.90,8.9,NaN,NaN
93,4,21,The One with the Invitation,Peter Bonerz,Seth Kurland,1998-04-23,21.50,7.2,NaN,NaN
142,6,22,The One Where Paul's the Man,Gary Halvorson,Story by: Brian CaldirolaTeleplay by: Sherry B...,2000-05-04,20.00,9.0,Brian Caldirola,Sherry Bilsing-Graham & Ellen Plummer
8,1,9,The One Where Underdog Gets Away,James Burrows,Jeff Greenstein & Jeff Strauss,1994-11-17,23.10,8.2,NaN,NaN
130,6,10,The One with the Routine,Kevin S. Bright,Brian Boyle,1999-12-16,22.40,8.6,NaN,NaN
56,3,9,The One with the Football,Kevin S. Bright,Ira Ungerleider,1996-11-21,29.30,9.0,NaN,NaN
51,3,4,The One with the Metaphorical Tunnel,Steve Zuckerman,Alexa Junge,1996-10-10,26.10,8.1,NaN,NaN
